# NB12 — Mini Project and Assessment

**Module 16 — Research Software Engineering**

---

## Overview

This notebook has two parts:

1. **MP01 integration demo**: demonstrate that `reverse_complement`, `gc_content`, and `kmer_frequencies` all work via import from the package directory (simulated here in-notebook since the package isn't installed in this environment).
2. **Assessment (5 stubs)**: five functions to implement. Each tests a core RSE skill from a different notebook in this module. Implement all five without looking at the reference implementations (in the HTML comment at the end).

---

## Part 1: MP01 — compbio_utils Integration Demo

In [ ]:
# Simulate 'from compbio_utils.sequence import ...' by defining the functions inline
# In the real MP01, you would: pip install -e utilities/compbio_utils/ and then import

from __future__ import annotations


COMPLEMENT_TABLE = str.maketrans('ATCGatcg', 'TAGCtagc')


def reverse_complement(seq: str) -> str:
    """Return the reverse complement of a DNA sequence."""
    seq_upper = seq.upper()
    invalid = set(seq_upper) - frozenset('ATCG')
    if invalid:
        raise ValueError(f'Invalid nucleotide(s): {", ".join(sorted(invalid))}')
    return seq_upper.translate(COMPLEMENT_TABLE)[::-1]


def gc_content(seq: str) -> float:
    """Compute GC content as a fraction in [0.0, 1.0]."""
    if not seq:
        return 0.0
    seq_upper = seq.upper()
    return (seq_upper.count('G') + seq_upper.count('C')) / len(seq_upper)


def kmer_frequencies(seq: str, k: int) -> dict[str, int]:
    """Compute k-mer frequencies."""
    seq_upper = seq.upper()
    if k <= 0 or k > len(seq_upper):
        raise ValueError(f'k={k} out of range for sequence of length {len(seq_upper)}')
    counts: dict[str, int] = {}
    for i in range(len(seq_upper) - k + 1):
        kmer = seq_upper[i:i + k]
        counts[kmer] = counts.get(kmer, 0) + 1
    return counts


# ---- MP01 Demonstration ----
print('=== MP01: compbio_utils Integration Demo ===')
print()

test_sequence = 'ATCGATCGATCGGGGGCCCCC'

rc = reverse_complement(test_sequence)
print(f'Input sequence:     {test_sequence}')
print(f'Reverse complement: {rc}')

gc = gc_content(test_sequence)
print(f'GC content:         {gc:.2%}')

kmer3 = kmer_frequencies(test_sequence, k=3)
top3 = sorted(kmer3.items(), key=lambda x: -x[1])[:5]
print(f'Top 5 trinucleotides: {top3}')

print()
print('All compbio_utils functions operational.')

## Part 2: Assessment — 5 Stubs

**Instructions:** Implement each function. The `raise NotImplementedError` line is your prompt. Do not look at the reference implementations until you have attempted all five.

**Rules:**
- Write the implementation yourself.
- Each function must pass the test harness cell beneath it.
- All five are graded independently.

---

### Q1: pytest conftest fixture (NB02)

In [ ]:
import pathlib

def write_conftest_fixture(tmp_path: pathlib.Path) -> pathlib.Path:
    """Write a temporary FASTA file and return its path.

    This simulates what a pytest fixture decorated with @pytest.fixture
    and using the built-in tmp_path fixture would do.

    The FASTA file must contain exactly two sequences:
      - '>seq1' with sequence 'ATCGATCG'
      - '>seq2' with sequence 'GCGCGCGC'

    Parameters
    ----------
    tmp_path : pathlib.Path
        Temporary directory (provided by pytest's tmp_path fixture).

    Returns
    -------
    pathlib.Path
        Path to the written FASTA file (e.g. tmp_path / 'test.fa').
    """
    raise NotImplementedError('Q1: conftest fixture')

In [ ]:
# Q1 Test harness
import tempfile

try:
    tmp = pathlib.Path(tempfile.mkdtemp())
    fasta_path = write_conftest_fixture(tmp)
    content = fasta_path.read_text()
    assert '>seq1' in content, 'Missing >seq1 header'
    assert '>seq2' in content, 'Missing >seq2 header'
    assert 'ATCGATCG' in content, 'Missing seq1 sequence'
    assert 'GCGCGCGC' in content, 'Missing seq2 sequence'
    assert fasta_path.exists(), 'File does not exist'
    print('Q1: PASS')
except NotImplementedError:
    print('Q1: NOT IMPLEMENTED')
except AssertionError as e:
    print(f'Q1: FAIL — {e}')

### Q2: Type annotation (NB06)

In [ ]:
def annotate_function_signature(fn_source: str) -> str:
    """Add type annotations to a Python function signature.

    Given source code for a function that takes one parameter named 'seq'
    and returns a float, rewrite the signature to add:
      - 'seq: str' for the parameter
      - '-> float' as the return type

    You may assume the input always has the pattern:
        def <name>(seq):

    Parameters
    ----------
    fn_source : str
        Python source code as a string. Contains exactly one function
        definition with a single untyped parameter 'seq'.

    Returns
    -------
    str
        Source code with the signature annotated:
        'def <name>(seq: str) -> float:'

    Examples
    --------
    >>> src = 'def gc(seq):\n    return 0.5'
    >>> annotate_function_signature(src)
    'def gc(seq: str) -> float:\n    return 0.5'
    """
    raise NotImplementedError('Q2: type annotation')

In [ ]:
# Q2 Test harness
try:
    src1 = 'def gc(seq):\n    return 0.5'
    result1 = annotate_function_signature(src1)
    assert 'seq: str' in result1, f'Missing seq: str in: {result1!r}'
    assert '-> float' in result1, f'Missing -> float in: {result1!r}'
    assert result1.startswith('def gc(seq: str) -> float:'), f'Wrong signature: {result1!r}'

    src2 = 'def compute_gc(seq):\n    pass'
    result2 = annotate_function_signature(src2)
    assert 'seq: str' in result2
    assert '-> float' in result2
    print('Q2: PASS')
except NotImplementedError:
    print('Q2: NOT IMPLEMENTED')
except AssertionError as e:
    print(f'Q2: FAIL — {e}')

### Q3: Cyclomatic complexity (NB06 / NB07)

In [ ]:
def compute_cyclomatic_complexity(fn_source: str) -> int:
    """Compute the cyclomatic complexity of a Python function.

    Cyclomatic complexity = 1 + number of decision points.
    Decision points are: if, elif, for, while, and, or, except, with
    (count each occurrence as +1).

    This is a simplified version of the McCabe metric, sufficient for
    the purposes of this assessment.

    Parameters
    ----------
    fn_source : str
        Python source code of a function as a string.

    Returns
    -------
    int
        Cyclomatic complexity (always >= 1).

    Examples
    --------
    >>> src = 'def f(x):\n    return x'  # no branches
    >>> compute_cyclomatic_complexity(src)
    1
    >>> src2 = 'def f(x):\n    if x > 0:\n        return 1\n    return 0'
    >>> compute_cyclomatic_complexity(src2)
    2
    """
    raise NotImplementedError('Q3: cyclomatic complexity')

In [ ]:
# Q3 Test harness
try:
    src_simple = 'def f(x):\n    return x'
    assert compute_cyclomatic_complexity(src_simple) == 1, \
        f'Expected 1, got {compute_cyclomatic_complexity(src_simple)}'

    src_one_if = 'def f(x):\n    if x > 0:\n        return 1\n    return 0'
    assert compute_cyclomatic_complexity(src_one_if) == 2, \
        f'Expected 2, got {compute_cyclomatic_complexity(src_one_if)}'

    src_for_if = 'def f(seq):\n    for c in seq:\n        if c == "G":\n            pass'
    cc = compute_cyclomatic_complexity(src_for_if)
    assert cc == 3, f'Expected 3, got {cc}'
    print('Q3: PASS')
except NotImplementedError:
    print('Q3: NOT IMPLEMENTED')
except AssertionError as e:
    print(f'Q3: FAIL — {e}')

### Q4: pyproject.toml generator (NB11)

In [ ]:
def generate_pyproject_toml(name: str, version: str, deps: list[str]) -> str:
    """Generate a minimal pyproject.toml string.

    The generated TOML must include:
    - [build-system] section with requires = ['hatchling'] and
      build-backend = 'hatchling.build'
    - [project] section with 'name', 'version', and 'dependencies' keys

    Parameters
    ----------
    name : str
        Package name (e.g. 'my-package').
    version : str
        Package version string (e.g. '0.1.0').
    deps : list[str]
        List of dependency strings (e.g. ['numpy>=1.24', 'pandas>=2.0']).

    Returns
    -------
    str
        A valid TOML string that, when parsed with tomllib, produces a dict
        where d['project']['name'] == name, d['project']['version'] == version,
        and d['project']['dependencies'] == deps.

    Examples
    --------
    >>> toml_str = generate_pyproject_toml('mylib', '1.0.0', ['numpy>=1.24'])
    >>> import tomllib
    >>> d = tomllib.loads(toml_str)
    >>> d['project']['name']
    'mylib'
    """
    raise NotImplementedError('Q4: pyproject.toml generator')

In [ ]:
# Q4 Test harness
import tomllib

try:
    result = generate_pyproject_toml(
        name='mylib',
        version='1.0.0',
        deps=['numpy>=1.24', 'pandas>=2.0'],
    )
    assert isinstance(result, str), 'Result must be a string'
    parsed = tomllib.loads(result)
    assert parsed['project']['name'] == 'mylib', \
        f"name mismatch: {parsed['project']['name']!r}"
    assert parsed['project']['version'] == '1.0.0', \
        f"version mismatch: {parsed['project']['version']!r}"
    assert parsed['project']['dependencies'] == ['numpy>=1.24', 'pandas>=2.0'], \
        f"deps mismatch: {parsed['project']['dependencies']}"
    assert 'build-system' in parsed, 'Missing [build-system] section'
    print('Q4: PASS')
except NotImplementedError:
    print('Q4: NOT IMPLEMENTED')
except AssertionError as e:
    print(f'Q4: FAIL — {e}')
except Exception as e:
    print(f'Q4: ERROR — {type(e).__name__}: {e}')

### Q5: Benchmark two implementations (NB07)

In [ ]:
from collections.abc import Callable
from typing import Any

def benchmark_two_implementations(
    fn_slow: Callable[..., Any],
    fn_fast: Callable[..., Any],
    inputs: list[Any],
    n_repeats: int = 5,
) -> dict[str, Any]:
    """Benchmark two function implementations on a list of inputs.

    For each input in ``inputs``, time ``fn_slow`` and ``fn_fast``
    using ``timeit.timeit`` with ``number=n_repeats``.
    Average the time per call in milliseconds.

    Parameters
    ----------
    fn_slow : Callable
        The 'slow' implementation to benchmark.
    fn_fast : Callable
        The 'fast' implementation to benchmark.
    inputs : list
        List of inputs to pass to each function (one at a time).
    n_repeats : int, optional
        Number of timeit repetitions per input. Default is 5.

    Returns
    -------
    dict[str, Any]
        Dictionary with keys:
        - 'slow_times_ms' : list[float] — average time per call in ms for fn_slow
        - 'fast_times_ms' : list[float] — average time per call in ms for fn_fast
        - 'speedups'      : list[float] — slow_time / fast_time for each input
        - 'mean_speedup'  : float — mean of speedups

    Examples
    --------
    >>> def slow_double(x): return sum(range(x))
    >>> def fast_double(x): return x * (x - 1) // 2
    >>> result = benchmark_two_implementations(slow_double, fast_double, [100, 1000])
    >>> result['mean_speedup'] > 1  # fast should be faster
    True
    """
    raise NotImplementedError('Q5: benchmark')

In [ ]:
# Q5 Test harness
try:
    def slow_sum(n: int) -> int:
        return sum(range(n))

    def fast_sum(n: int) -> int:
        return n * (n - 1) // 2

    result = benchmark_two_implementations(slow_sum, fast_sum, [1000, 5000, 10000], n_repeats=3)

    assert 'slow_times_ms' in result, 'Missing slow_times_ms'
    assert 'fast_times_ms' in result, 'Missing fast_times_ms'
    assert 'speedups' in result, 'Missing speedups'
    assert 'mean_speedup' in result, 'Missing mean_speedup'
    assert len(result['slow_times_ms']) == 3, f'Expected 3 times, got {len(result["slow_times_ms"])}'
    assert result['mean_speedup'] > 1.0, f'fast_sum should be faster, got speedup={result["mean_speedup"]:.2f}'
    print(f'Q5: PASS (mean speedup: {result["mean_speedup"]:.1f}x)')
except NotImplementedError:
    print('Q5: NOT IMPLEMENTED')
except AssertionError as e:
    print(f'Q5: FAIL — {e}')

## Assessment Summary

In [ ]:
print('=== Assessment Summary ===')
print()
print('Run the test harness cells above and record your results here:')
print()
print('Q1 (conftest fixture):      [ PASS / FAIL / NOT IMPLEMENTED ]')
print('Q2 (type annotation):       [ PASS / FAIL / NOT IMPLEMENTED ]')
print('Q3 (cyclomatic complexity): [ PASS / FAIL / NOT IMPLEMENTED ]')
print('Q4 (pyproject.toml):        [ PASS / FAIL / NOT IMPLEMENTED ]')
print('Q5 (benchmark):             [ PASS / FAIL / NOT IMPLEMENTED ]')

## Reference Implementations

<!-- REFERENCE IMPLEMENTATIONS (do not read until you have attempted all 5)

Q1:
def write_conftest_fixture(tmp_path):
    fasta_path = tmp_path / 'test.fa'
    fasta_path.write_text('>seq1\nATCGATCG\n>seq2\nGCGCGCGC\n')
    return fasta_path

Q2:
import re
def annotate_function_signature(fn_source):
    return re.sub(
        r'def (\w+)\(seq\):',
        r'def \1(seq: str) -> float:',
        fn_source
    )

Q3:
import ast
DECISION_NODES = (ast.If, ast.For, ast.While, ast.ExceptHandler,
                  ast.AsyncFor, ast.AsyncWith, ast.With,
                  ast.BoolOp)  # BoolOp covers 'and'/'or'
def compute_cyclomatic_complexity(fn_source):
    tree = ast.parse(fn_source)
    count = 1
    for node in ast.walk(tree):
        if isinstance(node, DECISION_NODES):
            if isinstance(node, ast.BoolOp):
                count += len(node.values) - 1
            else:
                count += 1
    return count

Q4:
def generate_pyproject_toml(name, version, deps):
    deps_toml = ', '.join(f'"{d}"' for d in deps)
    return f'''[build-system]
requires = ["hatchling"]
build-backend = "hatchling.build"

[project]
name = "{name}"
version = "{version}"
dependencies = [{deps_toml}]
'''

Q5:
import timeit
import numpy as np
def benchmark_two_implementations(fn_slow, fn_fast, inputs, n_repeats=5):
    slow_times, fast_times, speedups = [], [], []
    for inp in inputs:
        t_slow = timeit.timeit(lambda i=inp: fn_slow(i), number=n_repeats) / n_repeats * 1000
        t_fast = timeit.timeit(lambda i=inp: fn_fast(i), number=n_repeats) / n_repeats * 1000
        slow_times.append(t_slow)
        fast_times.append(t_fast)
        speedups.append(t_slow / t_fast if t_fast > 0 else float('inf'))
    return {
        'slow_times_ms': slow_times,
        'fast_times_ms': fast_times,
        'speedups': speedups,
        'mean_speedup': float(np.mean(speedups)),
    }

-->

## Reflection

*After completing the assessment:* Which of the five questions was hardest? Which RSE skill does it test? What would you study to strengthen the weakest area?

## Module 16 Completion Checklist

- [ ] NB01–NB12 completed and reflected on
- [ ] All 12 exercises attempted
- [ ] compbio_utils: sequence.py, stats.py, io.py implemented
- [ ] Test suite: ≥ 80% coverage
- [ ] ruff + mypy passing
- [ ] A01: 100% coverage + Sphinx docs
- [ ] A02: CLI k-mer search tool
- [ ] MP01: Published to TestPyPI